# NB 1.1 &mdash; Presa de contacte amb les dades

**MP 5134 &mdash; Disseny i avaluació de models basats en aprenentatge automàtic**
UT1: Entorn de treball i primer model de principi a fi

*Versió amb els pingüins de l'arxipèlag Palmer.*

---

### Què farem avui

Res d'entrenar models. Avui només mirem dades. Sembla poc, però és la meitat de
la feina real d'un projecte d'aprenentatge automàtic: si no entens què hi ha
dins del fitxer, qualsevol model que entrenis serà una loteria.

En acabar aquest notebook has de saber respondre:

1. Quantes mostres i quantes característiques té el nostre conjunt de dades?
2. Què volem predir exactament?
3. Quines files tenen valors absents, i què hi farem?
4. Què hi pinten, aquí, tres columnes que no són números?

## 1. De què va tot això

L'aprenentatge automàtic no és més que això: **tenim exemples del passat i volem
fer prediccions sobre casos nous**.

La diferència amb la programació que ja coneixeu és on posem les regles.

| Programació clàssica | Aprenentatge automàtic |
|---|---|
| Tu escrius les regles | L'algorisme les dedueix dels exemples |
| Dades + regles &rarr; resposta | Dades + respostes &rarr; regles |

Si volguéssim endevinar l'espècie d'un pingüí amb programació clàssica, hauríem
d'escriure nosaltres les condicions: *si el bec fa més de 45 mm i l'aleta menys
de 200, llavors...*. Amb aprenentatge automàtic li donem centenars de pingüins
ja identificats i deixem que l'algorisme trobi el patró.

Això té una conseqüència important: **el model només serà tan bo com les dades
que li donem**. D'aquí que avui dediquem tota una sessió a mirar-les.

## 2. El nostre conjunt de dades

Farem servir mesures de **344 pingüins** de tres illes de l'arxipèlag Palmer, a
l'Antàrtida, preses entre 2007 i 2009 per l'equip de l'estació de recerca Palmer
Station (Long Term Ecological Research). Cada fila és **un pingüí concret**,
mesurat amb un peu de rei i una balança.

Són mesures de camp: algú va anar fins allà, va agafar l'animal i el va mesurar.
Això importa més del que sembla, i ho veurem al llarg del curs.

| Columna | Què és |
|---|---|
| `species` | espècie: Adelie, Gentoo o Chinstrap |
| `island` | illa on es va trobar: Biscoe, Dream o Torgersen |
| `bill_length_mm` | llargada del bec (mm) |
| `bill_depth_mm` | gruix del bec (mm) |
| `flipper_length_mm` | llargada de l'aleta (mm) |
| `body_mass_g` | massa corporal (g) |
| `sex` | sexe: `male` o `female` |
| `year` | any de la campanya: 2007, 2008 o 2009 |

Les dades originals són de Gorman, Williams i Fraser (2014) i les distribueix el
paquet `palmerpenguins`. Aquí en tenim una còpia dins del repositori del mòdul,
perquè el notebook funcioni encara que el projecte original canviï d'adreça.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Còpia de les dades de palmerpenguins (Gorman et al., 2014) al repositori del mòdul
URL_DADES = "https://raw.githubusercontent.com/pprohenspolitecnicllevant/disseny-avaluacio-models-ml/refs/heads/main/UT01-Entorn_de_treball_primer_model/penguins/penguins.csv"

# read_csv() llegeix un fitxer de text separat per comes i el converteix en un
# DataFrame: la taula amb què treballa Pandas. Accepta una ruta local o una URL.
df = pd.read_csv(URL_DADES)

print("Dades carregades correctament.")

Fixa't en una cosa d'aquesta cel·la.

`pd.read_csv` accepta directament una URL: no cal descarregar res a mà. Això fa
que el notebook funcioni igual a qualsevol ordinador amb connexió, i que tots
treballem exactament amb el mateix fitxer.

Pandas endevina tot sol el tipus de cada columna a partir del contingut. De
vegades s'equivoca, i quan passa ho hem de corregir nosaltres; d'aquí a dues
cel·les comprovarem si aquest cop ho ha encertat.

### 2.1 Quantes dades tenim

El primer que es mira sempre d'un conjunt de dades és la seva forma.

In [ ]:
# .shape és un atribut (no porta parèntesis) amb la mida de la taula:
# una tupla (files, columnes) que aquí desempaquetem en dues variables.
files, columnes = df.shape

print(f"Files:    {files}")
print(f"Columnes: {columnes}")

Aquí apareix el **vocabulari fonamental** del mòdul. Val la pena fixar-lo ara,
perquè el farem servir cada dia fins al maig:

- **Mostra** o **instància**: una fila. En el nostre cas, un pingüí concret.
- **Característica** (*feature*): una columna que fem servir com a entrada.
- **Variable objectiu** (*target*): la columna que volem predir.

Una forma útil de recordar-ho: les mostres són *els exemples*, les
característiques són *el que sabem de cada exemple*, i la variable objectiu és
*el que volem endevinar*.

Compte amb una confusió molt freqüent: **no totes les columnes són
característiques**. Quina serà la variable objectiu ho decidirem nosaltres a la
secció 5, i aquella columna no entrarà mai com a entrada del model.

Tres-centes quaranta-quatre files són poques. Tan poques que aquest notebook les
podria mostrar totes per pantalla. Això, avui, és un avantatge: podem mirar les
dades de veritat, una per una si cal, en comptes de confiar en un resum. Ja
tornarem sobre la mida a la UT10.

In [ ]:
# head() retorna les primeres files. Per defecte cinc; head(20) en donaria vint.
# El seu germà tail() dóna les últimes.
df.head()

`head()` mostra les cinc primeres files. És el gest més repetit de tot el curs:
abans de fer res amb unes dades, mira-les.

In [ ]:
# info() no retorna res, imprimeix un resum: tipus de cada columna, quants
# valors no nuls té i la memòria que ocupa la taula.
df.info()

`info()` dóna tres coses de cop:

- el **tipus** de cada columna (`float64`, `int64`, `object`),
- quants valors **no nuls** té cadascuna,
- la memòria que ocupa.

Aquí ja hi ha dues coses per comentar.

Les quatre columnes de mesures són `float64`, que és el que esperàvem: Pandas
les ha llegit bé. Si una columna que hauria de ser numèrica aparegués com a
`object`, voldria dir que hi ha trobat text i no ha pogut convertir-la. És un
dels errors més habituals quan es carreguen dades europees, perquè sovint fan
servir la coma com a separador decimal.

I mira la columna de valors no nuls: no totes diuen 344. Aquest és el tema de la
secció 3, i és el primer problema real que ens trobarem.

In [ ]:
# describe() calcula els estadístics bàsics de cada columna NUMÈRICA:
# recompte, mitjana, desviació típica, mínim, els tres quartils i màxim.
# Les columnes de text les ignora (per veure-les hi ha value_counts(), secció 4).
df.describe()

### 2.2 Llegir un `describe()`

Aquesta taula sembla àrida però diu moltíssim. Mira-la amb aquestes preguntes:

**El `count` és el que toca?** Les quatre columnes de mesures diuen 342, no 344.
Ja tenim confirmat que hi ha dades que falten: `describe()` només compta els
valors que existeixen.

**Els mínims i màxims tenen sentit?** Els becs van de 32 a 60 mm i les aletes de
172 a 231 mm. Els pingüins pesen entre 2,7 i 6,3 kg. Són xifres raonables per a
un animal d'aquesta mida. Un pingüí de 10 kg o amb una aleta de 30 cm hauria de
fer-nos aixecar la cella: o és una espècie que no toca, o és un error de
transcripció.

**La mitjana i la mediana (50%) s'assemblen?** En la massa corporal, la mitjana
és lleugerament superior a la mediana. És una pista d'alguna cosa que veurem
dibuixada a la secció 6.

**Els quartils estan on t'esperes?** El 25% i el 75% et diuen com es reparteixen
els valors sense necessitat de dibuixar res.

I fixa't en el que **no** hi surt: `species`, `island` i `sex` no apareixen a la
taula. `describe()` només resumeix columnes numèriques, i aquestes tres no ho
són. Tornarem a buscar-les a la secció 4.

## 3. Valors absents

Cap conjunt de dades real està complet. Sensors que fallen, mesures que no es van
poder prendre, fitxes que es van quedar a mitges.

In [ ]:
# isna() torna una taula de la mateixa mida plena de True/False: True on falta
# el valor. Com que True val 1 i False val 0, sum() els compta per columna.
absents = df.isna().sum()

# absents[absents > 0] filtra la sèrie i deixa només les columnes amb algun buit;
# sort_values(ascending=False) les ordena de més a menys.
print(absents[absents > 0].sort_values(ascending=False))
print()

# dropna() retorna una còpia de la taula sense les files que tenen algun buit.
# No modifica df: el DataFrame original es queda tal com estava.
print(f"Files sense cap valor absent: {df.dropna().shape[0]} de {df.shape[0]}")

Amb un conjunt tan petit ens podem permetre una cosa que amb milions de files
seria impossible: **mirar-les una per una**.

In [ ]:
# any(axis=1) mira cada FILA i respon "hi ha algun True?".
#   axis=1 -> recorre les columnes de cada fila
#   axis=0 -> recorreria les files de cada columna (el comportament per defecte)
# El resultat és una sèrie de True/False amb una entrada per fila, i posar-la
# dins de df[...] selecciona només les files marcades amb True.
df[df.isna().any(axis=1)]

Aquesta taula val més que qualsevol explicació teòrica sobre valors absents,
perquè els buits tenen dues formes ben diferents.

**Dos pingüins no tenen cap mesura.** Ni bec, ni aleta, ni massa, ni sexe. De
tota la fitxa només en va quedar l'espècie i l'illa. Sigui quin sigui el motiu,
d'aquests dos animals no en podem fer res: no hi ha res a aprendre'n.

**Nou pingüins tenen totes les mesures però no consta el sexe.** Aquests sí que
són aprofitables per a gairebé tot; només queden fora si el sexe és justament el
que volem predir o fer servir com a entrada.

Això és un problema pràctic immediat: **la majoria d'algorismes de scikit-learn
no accepten valors absents**. Si li passes una taula amb buits, peta.

Hi ha tres sortides possibles, i cadascuna té un cost:

1. **Eliminar les files** amb buits. Simple, però perds mostres. Amb 344 files,
   perdre'n 11 és perdre'n el 3%.
2. **Eliminar la columna** sencera. Perds informació potencialment útil.
3. **Imputar**: omplir els buits amb la mitjana, la mediana o un valor estimat.

Avui farem servir l'opció 1 perquè és la més directa. A la **UT3** hi tornarem
amb calma, perquè la decisió no és innocent: imputar malament pot fer que el
model aprengui coses que no són certes.

## 4. Les columnes que no són números

`describe()` se n'ha desentès, però tres de les vuit columnes són text. Per
mirar-les fem servir una altra eina: comptar quantes vegades apareix cada valor.

In [ ]:
# value_counts() compta quantes vegades apareix cada valor diferent d'una
# columna, ordenat de més a menys freqüent. És l'equivalent de describe()
# per a les columnes de text.
# Amb normalize=True donaria proporcions en comptes de recomptes,
# i amb dropna=False comptaria també els valors absents.
for columna in ["species", "island", "sex"]:
    print(df[columna].value_counts())
    print()

Tres coses per llegir aquí.

**Les espècies no estan repartides per igual.** Els Adelie són gairebé la meitat
de les mostres i els Chinstrap prou menys d'una quarta part. Aquest desequilibri
té nom &mdash; **desbalanç de classes** &mdash; i conseqüències que treballarem a
la **UT4**. De moment, guarda't la proporció.

**El sexe està repartit gairebé al 50%.** 168 mascles i 165 femelles. Sumen 333,
no 344: els onze que falten són els nou de la secció anterior més els dos
pingüins sense cap mesura. Fixa't que `value_counts()` no els ha comptat enlloc,
simplement els ha ignorat en silenci. Aquest sí que està equilibrat, i el contrast amb l'espècie et
servirà per veure que el desbalanç no és una propietat de les dades en general,
sinó de cada columna en concret.

**I el més important: aquests valors són text, i cap model els accepta.**
`LinearRegression` no sap què fer amb la paraula `Adelie`. Perquè una columna
categòrica pugui entrar en un model l'hem de convertir en números, i fer-ho bé
té més "chicha" del que sembla: si dius que Adelie és 0, Gentoo és 1 i Chinstrap és
2, li estàs dient al model que Gentoo està *entre* els altres dos, cosa que no
vol dir res.

La manera correcta de fer-ho es diu **one-hot** i és contingut de la **UT3**.
Avui, per esquivar el problema, farem servir només les columnes numèriques.

## 5. Què volem predir exactament

Aquí hi ha una decisió de disseny important, i vull que la vegis des del primer
dia.

Quan es parla de predicció, el primer que ve al cap és endevinar el futur: què
passarà demà a partir del que sabem avui. Aquí no hi ha futur. Aquí tenim un
pingüí damunt la taula i el que fem és **predir una cosa que no hem mesurat a
partir de les que sí**: quant pesa, o de quina espècie és.

Les dues coses són predicció, i val la pena que ho tinguis clar: predir no vol
dir necessàriament endevinar el futur, vol dir **omplir un buit amb informació
que no teníem**.

Podríem intentar predir la massa a partir de la massa, o l'espècie a partir de
l'espècie. Seria un exercici buit: encertaríem sempre i no hauríem après res. Per
això la variable que volem endevinar no pot ser mai, alhora, una de les entrades.

In [ ]:
# Amb una llista de noms entre claudàtors dobles seleccionem un subconjunt de
# columnes i en tornem una taula nova. Amb un sol nom i claudàtors simples
# (df["species"]) obtindríem una sola columna, que en Pandas es diu Series.
df[["species", "bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]].head(8)

### 5.1 Dos problemes sobre les mateixes dades

Amb aquest fitxer podem plantejar els dos grans tipus de problema de machine learning supervisat, i tot 
dos sobre les dades d'un pingüí:

- **Regressió**: predir `body_mass_g`, un valor continu, a partir de les tres
  mesures del cos. És la pregunta de sempre en morfometria: quant pesa un animal
  amb aquestes proporcions i característiques?
- **Classificació**: predir `species`, una etiqueta d'un conjunt tancat de tres
  valors (una classe). És literalment la feina d'un biòleg de camp: a partir de l'animal i els seus
  trets l'has d'identificar.

La diferència no és tècnica sinó conceptual: en regressió la resposta pot ser
qualsevol valor contínu d'un rang (un nombre); en classificació la resposta és una categoria d'un
conjunt tancat.

## 6. Primeres gràfiques

Les taules diuen molt, però els ulls detecten coses que cap `describe()` et
donarà. Aquí ja treballem amb la llibreria **Matplotlib** per generar tot tipus de representacions
gràfiques de dades i així poder veure detalls que poden ser importants a l'hora de seleccionar i entrenar un model.

In [ ]:
plt.figure(figsize=(9, 4))  # amplada i alçada de la gràfica, en polzades

# hist() dibuixa l'histograma. Li passem la columna sense els valors absents,
# perquè matplotlib no els sap dibuixar; bins=30 és el nombre d'intervals.
plt.hist(df["body_mass_g"].dropna(), bins=30, edgecolor="white", color="g")

plt.xlabel("Massa corporal (g)")
plt.ylabel("Nombre de pingüins")
plt.title("Distribució de la variable objectiu (Massa corporal)")
plt.show()

Un **histograma** reparteix els valors en intervals i compta quants n'hi ha a
cadascun. Serveix per veure la forma de la distribució.

Aquesta no té forma de campana: s'endevinen dos grups, un de pingüins al voltant
dels 3.700 g i un altre de més pesants. Per què? Pensa-hi abans de continuar.

In [ ]:
plt.figure(figsize=(9, 4))

# unique() dóna els valors diferents d'una columna, aquí les tres espècies.
for especie in df["species"].unique():
    # .loc[files, columna] selecciona per etiqueta: de les files que compleixen
    # la condició, ens quedem només amb la columna de la massa.
    massa = df.loc[df["species"] == especie, "body_mass_g"].dropna()

    # alpha és la transparència: sense ella, cada histograma taparia l'anterior.
    plt.hist(massa, bins=20, alpha=0.6, label=especie, edgecolor="white")

plt.xlabel("Massa corporal (g)")
plt.ylabel("Nombre de pingüins")
plt.title("La mateixa distribució, separada per espècie")
plt.legend()  # mostra la llegenda amb les etiquetes de label
plt.show()

Aquí està la resposta: **els Gentoo són molt més grossos**. Els Adelie i els
Chinstrap pesen pràcticament el mateix i se superposen; els Gentoo formen el
segon grup de pingüins més aïllat però amb més massa.

Això ens diu una cosa pràctica: si l'espècie explica tan bé la massa, un model
que hagi de predir la massa i no sàpiga l'espècie ho tindrà més difícil. I al
revés: la massa hauria de ser una pista útil per endevinar l'espècie. Ho
comprovarem al notebook següent.

In [ ]:
plt.figure(figsize=(7, 6))

for especie in df["species"].unique():
    # df["species"] == especie genera una sèrie de True/False, i passar-la dins
    # de df[...] ens deixa només les files d'aquesta espècie. És el mateix
    # mecanisme de filtratge que hem fet servir amb els valors absents.
    subconjunt = df[df["species"] == especie]

    # scatter() dibuixa un punt per fila; s és la mida del punt.
    plt.scatter(subconjunt["bill_length_mm"], subconjunt["bill_depth_mm"],
                s=14, alpha=0.7, label=especie)

plt.xlabel("Llargada del bec (mm)")
plt.ylabel("Gruix del bec (mm)")
plt.title("Els becs separen les espècies?")
plt.legend()
plt.show()

Un **mapa de dispersió** enfronta dues variables. Cada punt és un pingüí.

La primera lectura és òbvia: **els tres grups estan gairebé separats**. Amb només
dues mesures del bec, quasi es podria dibuixar una frontera a mà que digués de
quina espècie és cada animal. Això és un molt bon senyal per al classificador que
entrenarem pròximament.

La segona lectura és més subtil, i és de les que fan que valgui la pena dibuixar
les coses. Mira el núvol sencer, oblidant els colors: sembla que quan el bec és
més llarg, és més prim. Ara mira **dins de cada color per separat**: dins de cada
espècie, els becs més llargs són també els més gruixuts. La relació canvia de
signe segons si mires el conjunt o els grups.

No és un error de les dades ni un truc del dibuix: passa sovint i té nom propi.
Aquesta idea, la de mesurar quant es relacionen dues variables i quan aquesta
mesura ens enganya, és tot el contingut de la **UT3: la correlació**.

## 7. Exercicis

**1.** Quants pingüins es van mesurar cada any? I a cada illa? Fes servir
`value_counts()` sobre les columnes `year` i `island`.

**2.** Totes les espècies viuen a totes les illes? Pista: `pd.crosstab(df["species"], df["island"])`.
Si la resposta et sorprèn, pensa què implicaria per a un model que hagués de
predir l'espècie sabent l'illa.

**3.** Quin va ser el pingüí més pesant de tota la sèrie? I el del bec més llarg?
Pista: `idxmax()` et dóna la posició del màxim, i `df.loc[...]` et dóna aquella
fila.

**4.** Calcula la massa mitjana de cada espècie i dibuixa-la amb un gràfic de
barres. Pista: `df.groupby("species")["body_mass_g"].mean()`.

**5.** Repeteix el mapa de dispersió de la secció 6 canviant els eixos per
`flipper_length_mm` i `body_mass_g`. Separa igual de bé les espècies?

**6.** Pensa i escriu la resposta: si volguéssim predir el **sexe** d'un pingüí,
quines columnes creus que serien més útils? I creus que seria un problema més
fàcil o més difícil que endevinar l'espècie? Ho comprovarem d'aquí unes setmanes.